In [1]:
!pip install tensorflow pandas scikit-learn ipywidgets
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2, ResNet50, EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
import time
import numpy as np
import os
import ipywidgets as widgets
from IPython.display import display
from tensorflow.keras.preprocessing import image

     ---------------------------------------- 0.0/8.9 MB ? eta -:--:--
     ---------------------------------------- 0.0/8.9 MB ? eta -:--:--
     ---------------------------------------- 0.0/8.9 MB ? eta -:--:--
     ---------------------------------------- 0.1/8.9 MB 1.5 MB/s eta 0:00:06
      --------------------------------------- 0.1/8.9 MB 1.1 MB/s eta 0:00:09
      --------------------------------------- 0.1/8.9 MB 901.1 kB/s eta 0:00:10
      --------------------------------------- 0.1/8.9 MB 901.1 kB/s eta 0:00:10
      --------------------------------------- 0.2/8.9 MB 737.3 kB/s eta 0:00:12
      --------------------------------------- 0.2/8.9 MB 655.1 kB/s eta 0:00:14
     - -------------------------------------- 0.2/8.9 MB 655.6 kB/s eta 0:00:14
     - -------------------------------------- 0.3/8.9 MB 628.5 kB/s eta 0:00:14
     - -------------------------------------- 0.3/8.9 MB 589.5 kB/s eta 0:00:15
     - -------------------------------------- 0.3/8.9 MB 593.9 kB/s eta


[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# --- CONFIGURATION ---
CSV_FILE = 'cats_vs_dogs.csv'
IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

if not os.path.exists(CSV_FILE):
    print(f"{CSV_FILE} not found. Creating dataset from TensorFlow Datasets...")
    try:
        get_ipython().system('python create_csv.py')
    except Exception as e:
        raise RuntimeError(
            f"Dataset file '{CSV_FILE}' is missing and automatic creation failed. "
            "Please run 'create_csv.py' manually or check your working directory."
        ) from e

print("Loading data from CSV...")
# 1. Load the CSV
df = pd.read_csv(CSV_FILE)

# 2. Convert labels (cat=0, dog=1)
df['label_num'] = df['label'].apply(lambda x: 1 if x == 'dog' else 0)

# 3. Split into Train and Validation
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label_num'])

print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")

# 4. Create TensorFlow Dataset from CSV paths
def load_image(image_path, label):
    img = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_HEIGHT, IMG_WIDTH])
    img = img / 255.0  # Normalize
    return img, label

def create_dataset(df_input):
    ds = tf.data.Dataset.from_tensor_slices((df_input['filepath'].values, df_input['label_num'].values))
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(buffer_size=AUTOTUNE)
    return ds

train_ds = create_dataset(train_df)
val_ds = create_dataset(val_df)

cats_vs_dogs.csv not found. Creating dataset from TensorFlow Datasets...



I0000 00:00:1779351083.801987   12956 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1779351090.150479   12956 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]
Dl Completed...:   0%|          | 0/1 [00:00<?, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]
Dl Completed...: 100%|██████████| 1/1 [00:00<00:00, 305.48 url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]
Dl Completed...: 100%|██████████| 1/1 [00:00<00:00, 305.48 url/s]

Dl Completed...: 100%|██████████| 1/1 [00:00<00:00, 305.48 url/s]

Dl Size...: 100%|██████████| 824

FileNotFoundError: [Errno 2] No such file or directory: 'cats_vs_dogs.csv'

In [ ]:
def build_model(base_model_class, model_name):
    print(f"\n--- Building {model_name} ---")
    
    base = base_model_class(
        weights='imagenet', 
        include_top=False, 
        input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)
    )
    base.trainable = False  # Freeze base
    
    x = base.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(1024, activation='relu')(x)
    predictions = layers.Dense(1, activation='sigmoid')(x) 
    
    model = models.Model(inputs=base.input, outputs=predictions, name=model_name)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model